# Raw Canny Edge Detection: Development, Validation, and Test Evaluation

This notebook evaluates raw Canny Edge Detection for printed circuit board (PCB) defect inspection using the authoritative dataset split.

The raw baseline compares an aligned defect-free PCB reference image with a defective PCB image. Both images are converted to grayscale, an absolute difference image is generated, Gaussian smoothing is applied, and Canny Edge Detection is used to extract structural edges. External contours are converted directly into predicted bounding boxes.

The experiment is organised into three stages:

1. **Development (139 images)** — establish and analyse the raw Canny baseline, investigate failure modes, and perform threshold diagnostics.
2. **Validation (139 images)** — evaluate the frozen raw Canny configuration on unseen validation images without changing the parameters.
3. **Test (415 images)** — perform the final evaluation using exactly the same frozen configuration.

No morphological post-processing, contour-area filtering, region merging, or bounding-box merging is used in the final raw Canny configuration.

## 1. Environment and Project Setup

The required Python libraries, project paths, shared preprocessing utilities, Canny implementation, evaluation functions, and development runner are loaded below.

In [ ]:
import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "data" / "dataset_split.csv").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from algorithms.common import load_image
from algorithms.evaluation import evaluate_boxes, parse_voc_boxes
from algorithms.canny import detect_canny

from scripts.run_canny_development import (
    evaluate_records,
    load_development_rows,
    summarise,
    write_results,
)

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 40)

print("Environment ready")
print("Project root:", PROJECT_ROOT)
print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

## 2. Frozen Raw Canny Configuration

The raw Canny configuration used for the main evaluation is fixed as follows:

- Gaussian blur kernel: **5 × 5**
- Canny low threshold: **50**
- Canny high threshold: **150**
- IoU threshold: **0.50**
- Morphological processing: **None**
- Contour-area filtering: **None**
- Region/bounding-box merging: **None**

The 50/150 thresholds are treated as the original raw baseline rather than as optimised values. Threshold sensitivity is investigated later on the development split, but the final validation and test stages retain the raw 50/150 configuration for direct comparison with the other baseline algorithms.

In [ ]:
MANIFEST_PATH = PROJECT_ROOT / "data" / "dataset_split.csv"

RESULTS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "canny_development.csv"
)

SUMMARY_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "canny_development_summary.json"
)

BOXES_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "canny_development_boxes"
)

VALIDATION_RESULTS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "canny_validation.csv"
)

TEST_RESULTS_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "metrics"
    / "canny_test.csv"
)

IOU_THRESHOLD = 0.50

LOW_THRESHOLD = 50
HIGH_THRESHOLD = 150
BLUR_KERNEL_SIZE = 5

RUN_DEVELOPMENT = False
RUN_VALIDATION = True
RUN_TEST = True

baseline_configuration = {
    "Input": "Aligned reference and defective PCB pair",
    "Shared preprocessing": "Dimension validation and grayscale conversion",
    "Comparison": "Absolute difference",
    "Noise reduction": "5 × 5 Gaussian smoothing",
    "Edge detection": "Canny (low=50, high=150)",
    "Contour extraction": "External contours",
    "Post-processing": "None",
    "Evaluation IoU": IOU_THRESHOLD,
}

pd.Series(
    baseline_configuration,
    name="Setting"
).to_frame()

## 3. Dataset Split Verification

The authoritative manifest contains 693 aligned PCB images divided into:

- 139 development images,
- 139 validation images, and
- 415 test images.

The split is verified before any evaluation is performed.

In [ ]:
manifest_df = pd.read_csv(MANIFEST_PATH)

development_df = manifest_df.loc[
    manifest_df["split"].eq("development")
].copy()

validation_df = manifest_df.loc[
    manifest_df["split"].eq("validation")
].copy()

test_df = manifest_df.loc[
    manifest_df["split"].eq("test")
].copy()

assert len(manifest_df) == 693
assert len(development_df) == 139
assert len(validation_df) == 139
assert len(test_df) == 415

assert development_df["image_id"].is_unique
assert validation_df["image_id"].is_unique
assert test_df["image_id"].is_unique

split_counts = (
    manifest_df["split"]
    .value_counts()
    .reindex(["development", "validation", "test"])
)

development_class_counts = (
    development_df["defect_class"]
    .value_counts()
    .sort_index()
)

print("Total aligned images:", len(manifest_df))
print("Development images:", len(development_df))
print("Validation images:", len(validation_df))
print("Test images:", len(test_df))

display(split_counts.rename("Images").to_frame())
display(
    development_class_counts
    .rename("Development Images")
    .to_frame()
)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4)
)

split_counts.plot.bar(
    ax=axes[0]
)
axes[0].set_title("Authoritative Dataset Split")
axes[0].set_ylabel("Images")
axes[0].tick_params(axis="x", rotation=0)

development_class_counts.plot.bar(
    ax=axes[1]
)
axes[1].set_title("Development Class Distribution")
axes[1].set_ylabel("Images")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

# Part A — Development Evaluation

## 4. Execute or Reload the Development Experiment

The raw Canny development experiment uses the 139-image development split. Previously generated CSV and JSON outputs are reused by default.

In [ ]:
outputs_exist = (
    RESULTS_PATH.is_file()
    and SUMMARY_PATH.is_file()
)

if RUN_DEVELOPMENT or not outputs_exist:
    development_rows = load_development_rows(
        MANIFEST_PATH
    )

    results = evaluate_records(
        development_rows,
        IOU_THRESHOLD,
        workers=1,
        boxes_directory=BOXES_DIR,
    )

    summary = summarise(
        results,
        IOU_THRESHOLD,
    )

    write_results(
        results,
        RESULTS_PATH,
    )

    SUMMARY_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    SUMMARY_PATH.write_text(
        json.dumps(
            summary,
            indent=2,
            sort_keys=True,
        ) + "\n",
        encoding="utf-8",
    )

    print(
        "Raw Canny development experiment completed."
    )
else:
    print(
        "Using existing raw Canny development results."
    )

In [ ]:
results_df = pd.read_csv(
    RESULTS_PATH
)

summary = json.loads(
    SUMMARY_PATH.read_text(
        encoding="utf-8"
    )
)

assert len(results_df) == 139
assert set(results_df["split"]) == {"development"}
assert set(results_df["status"]) == {"success"}

assert summary["successful"] == 139
assert summary["errors"] == 0

print(
    "Verified successful records:",
    summary["successful"]
)

print(
    "Errors:",
    summary["errors"]
)

results_df.head()

## 5. Overall Development Performance

Object-level localisation is evaluated using predicted and ground-truth bounding boxes. A predicted region is counted as a true positive only when it achieves an IoU of at least 0.50 with a ground-truth defect region.

In [ ]:
overall = summary[
    "overall_box_metrics"
]

overall_metrics = pd.DataFrame(
    {
        "Metric": [
            "True Positives",
            "False Positives",
            "False Negatives",
            "Precision",
            "Recall",
            "F1-score",
            "Mean Matched IoU",
        ],
        "Value": [
            overall["true_positives"],
            overall["false_positives"],
            overall["false_negatives"],
            overall["precision"],
            overall["recall"],
            overall["f1_score"],
            overall["mean_image_iou"],
        ],
    }
)

overall_metrics

## 6. Computational Performance

Processing time excludes disk I/O and represents the raw Canny processing stages.

In [ ]:
runtime = summary[
    "runtime_ms"
]

runtime_table = pd.DataFrame(
    {
        "Metric": [
            "Mean processing time (ms)",
            "Standard deviation (ms)",
            "Minimum processing time (ms)",
            "Maximum processing time (ms)",
            "Frames per second (FPS)",
        ],
        "Value": [
            runtime["mean"],
            runtime["standard_deviation"],
            runtime["minimum"],
            runtime["maximum"],
            runtime["frames_per_second"],
        ],
    }
)

runtime_table.round(4)

## 7. Behaviour by Defect Category

Per-class analysis is used to examine whether the raw Canny detector behaves differently across the six PCB defect categories.

In [ ]:
class_metrics_df = pd.DataFrame(
    summary[
        "box_metrics_by_class"
    ]
).T

class_metrics_df.index.name = (
    "Defect Class"
)

class_metrics_df

In [ ]:
diagnostic_summary = (
    results_df
    .groupby("defect_class")
    .agg(
        images=("image_id", "size"),
        mean_predicted_boxes=("predicted_count", "mean"),
        median_predicted_boxes=("predicted_count", "median"),
        maximum_predicted_boxes=("predicted_count", "max"),
        mean_edge_percentage=("edge_pixel_percentage", "mean"),
        mean_runtime_ms=("processing_time_ms", "mean"),
    )
)

diagnostic_summary.round(6)

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

results_df.boxplot(
    column="predicted_count",
    by="defect_class",
    ax=axes[0],
    rot=30,
)
axes[0].set_yscale(
    "symlog",
    linthresh=1,
)
axes[0].set_title("Predicted Contour Regions")
axes[0].set_xlabel("Defect Class")
axes[0].set_ylabel("Predicted Boxes")

results_df.boxplot(
    column="edge_pixel_percentage",
    by="defect_class",
    ax=axes[1],
    rot=30,
)
axes[1].set_title("Detected Edge Pixel Percentage")
axes[1].set_xlabel("Defect Class")
axes[1].set_ylabel("Edge Pixels (%)")

results_df.boxplot(
    column="processing_time_ms",
    by="defect_class",
    ax=axes[2],
    rot=30,
)
axes[2].set_title("Processing Time")
axes[2].set_xlabel("Defect Class")
axes[2].set_ylabel("Milliseconds")

fig.suptitle("")
plt.tight_layout()
plt.show()

## 8. Representative Raw Canny Results

The complete raw Canny processing pipeline is visualised as:

1. Reference PCB,
2. Defective PCB,
3. Absolute difference,
4. Gaussian-smoothed difference,
5. Canny edge map, and
6. Ground-truth and predicted bounding-box overlay.

Ground-truth boxes are red and predicted boxes are green.

In [ ]:
def show_canny_result(
    image_id,
    source_df,
    results_source_df=None,
    maximum_drawn_boxes=500,
):

    manifest_row = source_df.loc[
        source_df["image_id"].eq(image_id)
    ].iloc[0]

    reference = load_image(
        PROJECT_ROOT
        / manifest_row["reference_path"]
    )

    defective = load_image(
        PROJECT_ROOT
        / manifest_row["image_path"]
    )

    ground_truth = parse_voc_boxes(
        PROJECT_ROOT
        / manifest_row["annotation_path"]
    )

    detection = detect_canny(
        reference,
        defective,
        low_threshold=LOW_THRESHOLD,
        high_threshold=HIGH_THRESHOLD,
        blur_kernel_size=BLUR_KERNEL_SIZE,
    )

    overlay = defective.copy()

    for box in ground_truth:
        cv2.rectangle(
            overlay,
            (
                int(box["xmin"]),
                int(box["ymin"]),
            ),
            (
                int(box["xmax"]),
                int(box["ymax"]),
            ),
            (0, 0, 255),
            4,
        )

    boxes_drawn = (
        len(detection.boxes)
        <= maximum_drawn_boxes
    )

    if boxes_drawn:
        for box in detection.boxes:
            cv2.rectangle(
                overlay,
                (
                    int(box["xmin"]),
                    int(box["ymin"]),
                ),
                (
                    int(box["xmax"]),
                    int(box["ymax"]),
                ),
                (0, 255, 0),
                2,
            )

    mean_iou = np.nan

    if results_source_df is not None:
        matched_rows = results_source_df.loc[
            results_source_df["image_id"].eq(image_id)
        ]

        if not matched_rows.empty and "mean_matched_iou" in matched_rows.columns:
            mean_iou = float(
                matched_rows.iloc[0]["mean_matched_iou"]
            )

    fig, axes = plt.subplots(
        1,
        6,
        figsize=(28, 5),
    )

    axes[0].imshow(
        cv2.cvtColor(
            reference,
            cv2.COLOR_BGR2RGB,
        )
    )
    axes[0].set_title("Reference PCB")

    axes[1].imshow(
        cv2.cvtColor(
            defective,
            cv2.COLOR_BGR2RGB,
        )
    )
    axes[1].set_title("Defective PCB")

    axes[2].imshow(
        detection.difference,
        cmap="gray",
    )
    axes[2].set_title("Absolute Difference")

    axes[3].imshow(
        detection.blurred_difference,
        cmap="gray",
    )
    axes[3].set_title(
        "Gaussian-Smoothed Difference"
    )

    axes[4].imshow(
        detection.edge_map,
        cmap="gray",
    )
    axes[4].set_title("Canny Edge Map")

    axes[5].imshow(
        cv2.cvtColor(
            overlay,
            cv2.COLOR_BGR2RGB,
        )
    )
    axes[5].set_title(
        "GT Red; Prediction Green"
        if boxes_drawn
        else "GT Red; Predictions Omitted"
    )

    for axis in axes:
        axis.axis("off")

    iou_text = (
        f"{mean_iou:.3f}"
        if not np.isnan(mean_iou)
        else "N/A"
    )

    fig.suptitle(
        f"{image_id} | "
        f"{manifest_row['defect_class']} | "
        f"Predictions={len(detection.boxes)} | "
        f"IoU={iou_text}"
    )

    plt.tight_layout()
    plt.show()

## 9. Severe Fragmentation Example

The development image with the largest predicted-region count is inspected as an example of over-detection and contour fragmentation.

In [ ]:
most_fragmented_image_id = (
    results_df
    .sort_values(
        "predicted_count",
        ascending=False,
    )
    .iloc[0]["image_id"]
)

print(
    "Most fragmented example:",
    most_fragmented_image_id
)

show_canny_result(
    most_fragmented_image_id,
    development_df,
    results_df,
)

## 10. Zero-Detection Analysis

The number of development images with no predicted contour regions is calculated to quantify under-detection.

In [ ]:
zero_detection_examples = (
    results_df.loc[
        results_df[
            "predicted_count"
        ].eq(0)
    ]
)

zero_detection_count = len(
    zero_detection_examples
)

zero_detection_percentage = (
    zero_detection_count
    / len(results_df)
    * 100
)

print(
    "Number of development images with zero predicted regions:",
    zero_detection_count
)

print(
    f"Percentage of development images with zero predictions: "
    f"{zero_detection_percentage:.2f}%"
)

display(
    zero_detection_examples[
        [
            "image_id",
            "defect_class",
            "predicted_count",
            "edge_pixel_percentage",
        ]
    ].head(10)
)

if not zero_detection_examples.empty:
    zero_image_id = (
        zero_detection_examples
        .iloc[0]["image_id"]
    )

    show_canny_result(
        zero_image_id,
        development_df,
        results_df,
    )

## 11. Ground-Truth Region Diagnostic

A representative zero-detection image (`01_mouse_bite_01`) is inspected to determine whether defect-related intensity differences are present inside the annotated regions.

In [ ]:
diagnostic_image_id = "01_mouse_bite_01"

manifest_row = development_df.loc[
    development_df["image_id"].eq(
        diagnostic_image_id
    )
].iloc[0]

reference = load_image(
    PROJECT_ROOT
    / manifest_row["reference_path"]
)

defective = load_image(
    PROJECT_ROOT
    / manifest_row["image_path"]
)

reference_gray = cv2.cvtColor(
    reference,
    cv2.COLOR_BGR2GRAY
)

defective_gray = cv2.cvtColor(
    defective,
    cv2.COLOR_BGR2GRAY
)

difference = cv2.absdiff(
    reference_gray,
    defective_gray
)

print("Reference shape:", reference.shape)
print("Defective shape:", defective.shape)
print("Mean absolute difference:", difference.mean())
print("Maximum difference:", difference.max())
print(
    "Pixels with difference > 10:",
    np.sum(difference > 10)
)
print(
    "Pixels with difference > 30:",
    np.sum(difference > 30)
)
print(
    "Pixels with difference > 50:",
    np.sum(difference > 50)
)

In [ ]:
ground_truth = parse_voc_boxes(
    PROJECT_ROOT
    / manifest_row["annotation_path"]
)

for index, box in enumerate(
    ground_truth,
    start=1
):

    x1 = int(box["xmin"])
    y1 = int(box["ymin"])
    x2 = int(box["xmax"])
    y2 = int(box["ymax"])

    defect_region = difference[
        y1:y2,
        x1:x2
    ]

    print(f"\nDefect {index}")
    print(
        "Region shape:",
        defect_region.shape
    )
    print(
        "Mean difference:",
        defect_region.mean()
    )
    print(
        "Maximum difference:",
        defect_region.max()
    )
    print(
        "Pixels > 10:",
        np.sum(defect_region > 10)
    )
    print(
        "Pixels > 30:",
        np.sum(defect_region > 30)
    )
    print(
        "Pixels > 50:",
        np.sum(defect_region > 50)
    )

## 12. Single-Image Canny Threshold Sensitivity Diagnostic

Six threshold pairs are compared on the representative mouse-bite image while keeping the Gaussian smoothing unchanged.

This is a diagnostic experiment only and does not modify the frozen raw Canny configuration.

In [ ]:
threshold_pairs = [
    (5, 15),
    (10, 30),
    (15, 45),
    (20, 60),
    (30, 90),
    (50, 150),
]

blurred = cv2.GaussianBlur(
    difference,
    (
        BLUR_KERNEL_SIZE,
        BLUR_KERNEL_SIZE,
    ),
    0
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 9)
)

for axis, (low, high) in zip(
    axes.flat,
    threshold_pairs
):

    edges = cv2.Canny(
        blurred,
        low,
        high
    )

    edge_pixels = np.count_nonzero(
        edges
    )

    axis.imshow(
        edges,
        cmap="gray"
    )

    axis.set_title(
        f"Canny {low}/{high}\n"
        f"Edge pixels = {edge_pixels}"
    )

    axis.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
threshold_roi_results = []

for low, high in threshold_pairs:

    edges = cv2.Canny(
        blurred,
        low,
        high
    )

    for index, box in enumerate(
        ground_truth,
        start=1
    ):

        x1 = int(box["xmin"])
        y1 = int(box["ymin"])
        x2 = int(box["xmax"])
        y2 = int(box["ymax"])

        roi_edges = edges[
            y1:y2,
            x1:x2
        ]

        threshold_roi_results.append(
            {
                "low_threshold": low,
                "high_threshold": high,
                "defect": index,
                "edge_pixels_in_gt": int(
                    np.count_nonzero(
                        roi_edges
                    )
                ),
            }
        )

threshold_roi_df = pd.DataFrame(
    threshold_roi_results
)

threshold_roi_df

In [ ]:
threshold_pivot = (
    threshold_roi_df
    .pivot(
        index=[
            "low_threshold",
            "high_threshold"
        ],
        columns="defect",
        values="edge_pixels_in_gt"
    )
)

threshold_pivot.columns = [
    f"Defect {column}"
    for column in threshold_pivot.columns
]

threshold_pivot.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title(
    "Edge Responses Within Ground-Truth Defect Regions"
)

plt.xlabel(
    "Canny Threshold Pair"
)

plt.ylabel(
    "Detected Edge Pixels"
)

plt.xticks(
    rotation=30
)

plt.tight_layout()
plt.show()

## 13. Full Development Threshold Tuning

The single-image diagnostic shows that lower thresholds preserve more edge responses. To determine whether this improves object-level localisation, five threshold pairs are evaluated across all 139 development images while keeping the Gaussian kernel fixed at 5 × 5 and using no post-processing.

This section is retained as development evidence only. The project subsequently chooses to keep the original 50/150 raw configuration for validation and final testing rather than introducing a separate enhanced Canny method.

In [ ]:
threshold_candidates = [
    (50, 150),
    (30, 90),
    (20, 60),
    (15, 45),
    (10, 30),
]

development_records = (
    development_df
    .to_dict("records")
)

threshold_comparison_results = []

for low, high in threshold_candidates:

    print(
        f"\nRunning Canny thresholds: "
        f"{low}/{high}"
    )

    total_tp = 0
    total_fp = 0
    total_fn = 0

    image_ious = []
    predicted_counts = []
    runtimes = []
    zero_detection_images = 0

    for index, row in enumerate(
        development_records,
        start=1
    ):

        reference = load_image(
            PROJECT_ROOT
            / row["reference_path"]
        )

        defective = load_image(
            PROJECT_ROOT
            / row["image_path"]
        )

        ground_truth_boxes = (
            parse_voc_boxes(
                PROJECT_ROOT
                / row["annotation_path"]
            )
        )

        detection = detect_canny(
            reference,
            defective,
            low_threshold=low,
            high_threshold=high,
            blur_kernel_size=BLUR_KERNEL_SIZE,
        )

        metrics = evaluate_boxes(
            detection.boxes,
            ground_truth_boxes,
            iou_threshold=IOU_THRESHOLD,
        )

        total_tp += int(
            metrics["true_positives"]
        )
        total_fp += int(
            metrics["false_positives"]
        )
        total_fn += int(
            metrics["false_negatives"]
        )

        image_ious.append(
            float(
                metrics["mean_matched_iou"]
            )
        )

        predicted_counts.append(
            len(detection.boxes)
        )

        runtimes.append(
            float(
                detection.processing_time_ms
            )
        )

        if len(detection.boxes) == 0:
            zero_detection_images += 1

        if (
            index % 25 == 0
            or index == len(
                development_records
            )
        ):
            print(
                f"Processed "
                f"{index}/"
                f"{len(development_records)}"
            )

    precision = (
        total_tp
        / (total_tp + total_fp)
        if total_tp + total_fp
        else 0.0
    )

    recall = (
        total_tp
        / (total_tp + total_fn)
        if total_tp + total_fn
        else 0.0
    )

    f1_score = (
        2.0
        * precision
        * recall
        / (
            precision
            + recall
        )
        if precision + recall
        else 0.0
    )

    threshold_comparison_results.append(
        {
            "low_threshold": low,
            "high_threshold": high,
            "true_positives": total_tp,
            "false_positives": total_fp,
            "false_negatives": total_fn,
            "precision": precision,
            "recall": recall,
            "f1_score": f1_score,
            "mean_iou": float(
                np.mean(image_ious)
            ),
            "zero_detection_images": (
                zero_detection_images
            ),
            "mean_predicted_boxes": float(
                np.mean(
                    predicted_counts
                )
            ),
            "mean_runtime_ms": float(
                np.mean(runtimes)
            ),
        }
    )

threshold_comparison_df = pd.DataFrame(
    threshold_comparison_results
)

threshold_comparison_df

In [ ]:
comparison_plot_df = (
    threshold_comparison_df
    .copy()
)

comparison_plot_df[
    "threshold_pair"
] = (
    comparison_plot_df[
        "low_threshold"
    ].astype(str)
    + "/"
    + comparison_plot_df[
        "high_threshold"
    ].astype(str)
)

comparison_plot_df = (
    comparison_plot_df
    .set_index(
        "threshold_pair"
    )
)

comparison_plot_df[
    [
        "precision",
        "recall",
        "f1_score",
        "mean_iou",
    ]
].plot(
    kind="bar",
    figsize=(11, 6)
)

plt.title(
    "Full Development Threshold Comparison"
)
plt.xlabel(
    "Canny Threshold Pair"
)
plt.ylabel(
    "Metric Value"
)
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5)
)

comparison_plot_df[
    "zero_detection_images"
].plot.bar(
    ax=axes[0]
)
axes[0].set_title(
    "Zero-Detection Images"
)
axes[0].set_xlabel(
    "Threshold Pair"
)
axes[0].set_ylabel(
    "Images"
)
axes[0].tick_params(
    axis="x",
    rotation=0
)

comparison_plot_df[
    "mean_predicted_boxes"
].plot.bar(
    ax=axes[1]
)
axes[1].set_title(
    "Mean Predicted Boxes per Image"
)
axes[1].set_xlabel(
    "Threshold Pair"
)
axes[1].set_ylabel(
    "Mean Predicted Boxes"
)
axes[1].tick_params(
    axis="x",
    rotation=0
)

plt.tight_layout()
plt.show()

## 14. Development Interpretation

The raw development experiment demonstrates two major limitations:

1. **Under-detection:** the 50/150 baseline produces many zero-detection images.
2. **Over-detection and fragmentation:** lowering the thresholds increases edge sensitivity but also creates many additional contour-derived bounding boxes and false positives.

The threshold experiment therefore shows that threshold adjustment alone is insufficient to convert raw Canny edges into reliable defect-localisation regions.

For the simplified project route used in this notebook, no advanced morphological enhancement is introduced. The original raw 50/150 configuration is retained as the frozen baseline for validation and test evaluation.

# Part B — Validation Evaluation

## 15. Validation of the Frozen Raw Canny Configuration

The frozen raw Canny configuration is now evaluated on the 139 validation images.

**Important:** No parameters are changed based on validation performance. The validation stage is used only to examine whether the raw baseline behaviour generalises beyond the development split.

In [ ]:
FROZEN_LOW_THRESHOLD = 50
FROZEN_HIGH_THRESHOLD = 150
FROZEN_BLUR_KERNEL_SIZE = 5

frozen_configuration = {
    "low_threshold": FROZEN_LOW_THRESHOLD,
    "high_threshold": FROZEN_HIGH_THRESHOLD,
    "blur_kernel_size": FROZEN_BLUR_KERNEL_SIZE,
    "iou_threshold": IOU_THRESHOLD,
    "post_processing": "None",
}

pd.Series(
    frozen_configuration,
    name="Frozen Setting"
).to_frame()

### 15.1 Split Evaluation Helper

The same helper is used for validation and test so that the evaluation procedure remains identical.

In [ ]:
def evaluate_manifest_split(
    split_df,
    split_name,
    low_threshold=FROZEN_LOW_THRESHOLD,
    high_threshold=FROZEN_HIGH_THRESHOLD,
    blur_kernel_size=FROZEN_BLUR_KERNEL_SIZE,
    iou_threshold=IOU_THRESHOLD,
):
    records = split_df.to_dict(
        "records"
    )

    rows = []

    for index, row in enumerate(
        records,
        start=1
    ):
        reference = load_image(
            PROJECT_ROOT
            / row["reference_path"]
        )

        defective = load_image(
            PROJECT_ROOT
            / row["image_path"]
        )

        ground_truth_boxes = (
            parse_voc_boxes(
                PROJECT_ROOT
                / row["annotation_path"]
            )
        )

        detection = detect_canny(
            reference,
            defective,
            low_threshold=low_threshold,
            high_threshold=high_threshold,
            blur_kernel_size=blur_kernel_size,
        )

        metrics = evaluate_boxes(
            detection.boxes,
            ground_truth_boxes,
            iou_threshold=iou_threshold,
        )

        edge_pixel_percentage = (
            float(
                np.count_nonzero(
                    detection.edge_map
                )
            )
            / detection.edge_map.size
            * 100.0
        )

        rows.append(
            {
                "image_id": row["image_id"],
                "defect_class": row["defect_class"],
                "split": split_name,
                "low_threshold": low_threshold,
                "high_threshold": high_threshold,
                "blur_kernel_size": blur_kernel_size,
                "predicted_count": len(
                    detection.boxes
                ),
                "ground_truth_count": len(
                    ground_truth_boxes
                ),
                **metrics,
                "edge_pixel_percentage": (
                    edge_pixel_percentage
                ),
                "processing_time_ms": (
                    detection.processing_time_ms
                ),
            }
        )

        if (
            index % 25 == 0
            or index == len(records)
        ):
            print(
                f"{split_name}: "
                f"Processed "
                f"{index}/{len(records)}"
            )

    return pd.DataFrame(rows)

In [ ]:
def summarise_split_results(
    split_results_df,
    split_name,
):

    tp = int(
        split_results_df[
            "true_positives"
        ].sum()
    )

    fp = int(
        split_results_df[
            "false_positives"
        ].sum()
    )

    fn = int(
        split_results_df[
            "false_negatives"
        ].sum()
    )

    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0.0
    )

    f1_score = (
        2.0
        * precision
        * recall
        / (
            precision
            + recall
        )
        if precision + recall
        else 0.0
    )

    zero_detection_images = int(
        (
            split_results_df[
                "predicted_count"
            ] == 0
        ).sum()
    )

    return {
        "split": split_name,
        "images": len(
            split_results_df
        ),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "mean_iou": float(
            split_results_df[
                "mean_matched_iou"
            ].mean()
        ),
        "zero_detection_images": (
            zero_detection_images
        ),
        "mean_predicted_boxes": float(
            split_results_df[
                "predicted_count"
            ].mean()
        ),
        "mean_runtime_ms": float(
            split_results_df[
                "processing_time_ms"
            ].mean()
        ),
        "fps": (
            1000.0
            / float(
                split_results_df[
                    "processing_time_ms"
                ].mean()
            )
            if float(
                split_results_df[
                    "processing_time_ms"
                ].mean()
            ) > 0
            else 0.0
        ),
    }

### 15.2 Run or Reload Validation Results

In [ ]:
if RUN_VALIDATION or not VALIDATION_RESULTS_PATH.is_file():

    validation_results_df = (
        evaluate_manifest_split(
            validation_df,
            "validation",
        )
    )

    VALIDATION_RESULTS_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    validation_results_df.to_csv(
        VALIDATION_RESULTS_PATH,
        index=False,
    )

    print(
        "Validation results saved to:",
        VALIDATION_RESULTS_PATH
    )

else:
    validation_results_df = pd.read_csv(
        VALIDATION_RESULTS_PATH
    )

    print(
        "Using existing validation results."
    )

assert len(validation_results_df) == 139

validation_results_df.head()

In [ ]:
validation_summary = (
    summarise_split_results(
        validation_results_df,
        "validation",
    )
)

validation_summary_df = pd.DataFrame(
    {
        "Metric": list(
            validation_summary.keys()
        ),
        "Value": list(
            validation_summary.values()
        ),
    }
)

validation_summary_df

### 15.3 Validation Performance by Defect Category

In [ ]:
validation_class_summary = (
    validation_results_df
    .groupby("defect_class")
    .agg(
        images=("image_id", "size"),
        true_positives=("true_positives", "sum"),
        false_positives=("false_positives", "sum"),
        false_negatives=("false_negatives", "sum"),
        mean_iou=("mean_matched_iou", "mean"),
        mean_predicted_boxes=("predicted_count", "mean"),
        mean_runtime_ms=("processing_time_ms", "mean"),
    )
)

validation_class_summary

# Part C — Final Test Evaluation

## 16. Final Test of the Frozen Raw Canny Configuration

The final test uses the 415-image test split.

The configuration remains exactly the same as the development and validation raw baseline:

- Canny: 50/150
- Gaussian blur: 5 × 5
- IoU threshold: 0.50
- No morphology
- No contour filtering
- No region or box merging

No further tuning is permitted after viewing the test results.

### 16.1 Run or Reload Test Results

In [ ]:
if RUN_TEST or not TEST_RESULTS_PATH.is_file():

    test_results_df = (
        evaluate_manifest_split(
            test_df,
            "test",
        )
    )

    TEST_RESULTS_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    test_results_df.to_csv(
        TEST_RESULTS_PATH,
        index=False,
    )

    print(
        "Test results saved to:",
        TEST_RESULTS_PATH
    )

else:
    test_results_df = pd.read_csv(
        TEST_RESULTS_PATH
    )

    print(
        "Using existing test results."
    )

assert len(test_results_df) == 415

test_results_df.head()

In [ ]:
test_summary = (
    summarise_split_results(
        test_results_df,
        "test",
    )
)

test_summary_df = pd.DataFrame(
    {
        "Metric": list(
            test_summary.keys()
        ),
        "Value": list(
            test_summary.values()
        ),
    }
)

test_summary_df

### 16.2 Test Performance by Defect Category

In [ ]:
test_class_summary = (
    test_results_df
    .groupby("defect_class")
    .agg(
        images=("image_id", "size"),
        true_positives=("true_positives", "sum"),
        false_positives=("false_positives", "sum"),
        false_negatives=("false_negatives", "sum"),
        mean_iou=("mean_matched_iou", "mean"),
        mean_predicted_boxes=("predicted_count", "mean"),
        mean_runtime_ms=("processing_time_ms", "mean"),
    )
)

test_class_summary

# Part D — Development, Validation, and Test Comparison

## 17. Overall Split Comparison

The raw Canny baseline is compared across all three dataset partitions to show whether the observed behaviour remains consistent from development through final testing.

In [ ]:
development_summary_for_comparison = {
    "split": "development",
    "images": len(results_df),
    "true_positives": int(
        overall["true_positives"]
    ),
    "false_positives": int(
        overall["false_positives"]
    ),
    "false_negatives": int(
        overall["false_negatives"]
    ),
    "precision": float(
        overall["precision"]
    ),
    "recall": float(
        overall["recall"]
    ),
    "f1_score": float(
        overall["f1_score"]
    ),
    "mean_iou": float(
        overall["mean_image_iou"]
    ),
    "zero_detection_images": int(
        (
            results_df[
                "predicted_count"
            ] == 0
        ).sum()
    ),
    "mean_predicted_boxes": float(
        results_df[
            "predicted_count"
        ].mean()
    ),
    "mean_runtime_ms": float(
        results_df[
            "processing_time_ms"
        ].mean()
    ),
    "fps": (
        1000.0
        / float(
            results_df[
                "processing_time_ms"
            ].mean()
        )
        if float(
            results_df[
                "processing_time_ms"
            ].mean()
        ) > 0
        else 0.0
    ),
}

split_comparison_df = pd.DataFrame(
    [
        development_summary_for_comparison,
        validation_summary,
        test_summary,
    ]
)

split_comparison_df

In [ ]:
plot_df = (
    split_comparison_df
    .set_index("split")
)

plot_df[
    [
        "precision",
        "recall",
        "f1_score",
        "mean_iou",
    ]
].plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title(
    "Raw Canny Performance Across Dataset Splits"
)
plt.xlabel(
    "Dataset Split"
)
plt.ylabel(
    "Metric Value"
)
plt.xticks(
    rotation=0
)
plt.ylim(
    0,
    1
)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 5)
)

plot_df[
    "zero_detection_images"
].plot.bar(
    ax=axes[0]
)
axes[0].set_title(
    "Zero-Detection Images by Split"
)
axes[0].set_xlabel(
    "Dataset Split"
)
axes[0].set_ylabel(
    "Images"
)
axes[0].tick_params(
    axis="x",
    rotation=0
)

plot_df[
    "mean_predicted_boxes"
].plot.bar(
    ax=axes[1]
)
axes[1].set_title(
    "Mean Predicted Boxes by Split"
)
axes[1].set_xlabel(
    "Dataset Split"
)
axes[1].set_ylabel(
    "Mean Predicted Boxes"
)
axes[1].tick_params(
    axis="x",
    rotation=0
)

plt.tight_layout()
plt.show()

## 18. Final Raw Canny Conclusion

The complete experiment evaluates raw Canny Edge Detection across the development, validation, and test splits using the same frozen configuration.

The development analysis demonstrates that raw Canny can detect structural intensity transitions efficiently, but the resulting edge contours do not necessarily correspond to complete annotated defect regions. The method therefore suffers from both under-detection and fragmentation.

Threshold sensitivity analysis further shows that lowering the Canny thresholds reduces zero-detection cases but substantially increases the number of predicted contour regions and false positives. Threshold adjustment alone is therefore insufficient to provide reliable object-level PCB defect localisation.

For this project, the raw 50/150 Canny configuration is retained without advanced morphological enhancement so that its standalone behaviour can be compared directly with Otsu Thresholding, Template Matching, and ORB Feature Matching.

The final interpretation should be based on the actual validation and test metrics generated above, particularly precision, recall, F1-score, mean IoU, false positives, false negatives, runtime, and zero-detection behaviour.